# Querying consDB and EFD for AOS

Owner: **Chris  Suberlak** [@suberlak](https://github.com/lsst-ts/ts_aos_analysis/issues/new?body=@suberlak) <br>
Last Verified to Run: **2024-11-05** <br>
Software Versions:
  - `lsst_distrib`: **w_2024_43**

## Setup:

This notebook was run on https://usdf-rsp.slac.stanford.edu/ .  The only necessary step required to access `consDB` is to remove the `HTTP_PROXY` environmental variable if present, with eg. `del os.environ["http_proxy"]`.

The Engineering Facilities Database, or EFD, contains fine-grained telemetry from a variety of sensors and instruments, such as 
force balance offsets (hardpoint correction), applied forces (sum of all above + AOS closed-loop corrections). For example [this movie](https://rubin-obs.slack.com/archives/C07QM715AJY/p1718056301907499) was prepared with the EFD data by C. Lage. 


ConsDB generally contains exposure-level  aggregate information, i.e. either information already present in the header of each exposure,  or  in the "Transformed EFD" portion of the ConsDB,  the aggregate  of the time-series EFD data over the exposure time window with the ["EFD Transformation service".](https://rubin-obs.slack.com/archives/C07QJMQ7L4A/p1716317310421029?thread_ts=1716315549.132149&cid=C07QJMQ7L4A)  

In [ ]:
from astropy.time import Time, TimeDelta
import pandas as pd
from lsst.summit.utils.efdUtils import getEfdData, makeEfdClient
from lsst.daf import butler as dafButler
import lsst.summit.utils.butlerUtils as butlerUtils
from lsst.summit.utils import ConsDbClient
from lsst.summit.utils.utils import computeCcdExposureId

## Resources on consDB 

* A grafana page with up-to-date [consDB status](https://grafana.slac.stanford.edu/d/z7FCA4Nnk/cloud-native-postgresql-cnpg?orgId=1&refresh=30s&var-DataSource=940RXge4k&var-vcluster=vcluster--usdf-summitdb&var-cluster=summit-db-replica&var-instances=All&var-namespace=summit-db-replica&var-resolution=5m&from=now-24h&to=now)

* Confluence page with documentation on [consDB access](https://rubinobs.atlassian.net/wiki/spaces/~ktl/pages/55377993/ConsDB+Usage) 

* [Schema browser](https://sdm-schemas.lsst.io/) 

* Tech-note on the [background for consDB](https://dmtn-227.lsst.io/) 

* [ConsDB transformed EFD topics](https://rubinobs.atlassian.net/wiki/spaces/DM/pages/48836130/Consolidated+Database+Transformed+EFD+Topics)



## Connecting to consolidated database (consDB)

In [ ]:
import os 
del os.environ["http_proxy"]

In [ ]:
client = ConsDbClient('http://consdb-pq.consdb:8080/consdb')
print(f'schemas:\n', client.schema())

In [ ]:
client = ConsDbClient('http://consdb-pq.consdb:8080/consdb')
print(client.schema())  # list the instruments
print(client.schema('lsstcomcam'))  # list tables for an instrument

In [ ]:
print(client.schema('lsstcomcam', 'cdb_lsstcomcam.ccdexposure_camera'))

In [ ]:
schema = 'lsstcomcam'
table = 'exposure'
print(f'columns (table={table}): {table} \n',
      list(client.schema('lsstcomcam', f'cdb_{schema}.{table}').keys()
          )
      )

In [ ]:
# https://rubin-obs.slack.com/archives/C07QJMQ7L4A/p1730392968242269?thread_ts=1730385376.310139&cid=C07QJMQ7L4A
# For investigating schema and table content 
client = ConsDbClient('http://consdb-pq.consdb:8080/consdb')
schema = 'lsstcomcam'
table = 'ccdvisit1_quicklook'
print(f'schemas:\n', client.schema())  # list the instruments
print(f'tables (schema={schema}):\n', client.schema(schema))  # list tables for an instrument
print(f'columns (table={table}): {table}\n', list(client.schema('lsstcomcam', f'cdb_{schema}.{table}').keys()))

In [ ]:
day_obs = 20241025
query = f"""
SELECT e.band, e.exp_time, q.* 
from cdb_lsstcomcam.visit1_quicklook q, cdb_lsstcomcam.exposure e
where q.visit_id = e.exposure_id
and e.day_obs = {day_obs}
--order by e.seq_num desc
"""
df = client.query(query)
#df.columns
#df[['visit_id', 'eff_time_median', 'eff_time_sky_bg_scale_median', 'eff_time_psf_sigma_scale_median', 'eff_time_zero_point_scale_median']]
df[['visit_id', 'band', 'psf_sigma_median', 'sky_bg_median', 'zero_point_median']][:5]

In [ ]:
day_obs = 20241025
query = f"""
SELECT v.band, v.exp_time, q.*
from cdb_lsstcomcam.visit1_quicklook q, cdb_lsstcomcam.visit1 v
where q.visit_id = v.visit_id
and v.day_obs = {day_obs}
"""
df = client.query(query)
df[:5]

## Resources on EFD

* [EFD documentation](https://sasquatch.lsst.io/user-guide/observatorytelemetry.html)
* [Query examples](https://github.com/lsst-sqre/system-test/tree/main/efd_examples)
* [Documentation for the python connection client](https://efd-client.lsst.io/api.html)


## Connecting to EFD 

In [ ]:
butler = dafButler.Butler('/repo/embargo_new', 
                          collections=["LSSTComCam/raw/all", 
                                       "LSSTComCam/calib", 
                                       "LSSTComCam/nightlyValidation"
                                                           ]
                         )
client = makeEfdClient()

Given eg. RubinTV, we can find two defocal pairs to query EFD for the time corresponding to just before the first exposure, and just after the last exposure. For instance, on  `20241030` seq_num `72` and `73`. One way to get the time spans is to query the registry, and look at `min`, `max` values of `dataId.exposure.timespan`

In [ ]:
dataRefs = list(butler.registry.queryDatasets('postISRCCD', where="instrument='LSSTComCam' and \
exposure.observation_type='cwfs' and day_obs = 20241030 and exposure.seq_num in (72,73)").expanded())

Or we can use the `butlerUtils` package to obtain the exposure record (including timespan) for the first and last seqNum:

In [ ]:
expId = 2024103000072
dataId = {'exposure': expId, 'detector': 4, 'instrument': 'LSSTComCam'}
expRecordBegin = butlerUtils.getExpRecordFromDataId(butler, dataId)

In [ ]:
expId = 2024103000073
dataId = {'exposure': expId, 'detector': 4, 'instrument': 'LSSTComCam'}
expRecordEnd = butlerUtils.getExpRecordFromDataId(butler, dataId)

In [ ]:
expRecordBegin.timespan.begin

In [ ]:
expRecordEnd.timespan.end

In [ ]:
td = expRecordEnd.timespan.end-expRecordBegin.timespan.begin
print('Time it took to take two exposures:', td.sec, ' sec') 

Which makes sense with 30 sec exposure time. 

A list of EFD topics is available 

In [ ]:
topics = await client.get_topics()
len(topics)

A huge number of topics is available, and those that start with `lsst.sal` are most pertinent:

In [ ]:
# from https://rubin-obs.slack.com/files/U07D8TLA3C3/F07QJPNTUE7/searching_the_efd
tt = [t.split('.') for t in topics]
topic_dict = {}
for t in tt:
    i = 0
    dd = topic_dict
    while i < len(t):
        if t[i] not in dd.keys():
            dd[t[i]] = {}
        dd = dd[t[i]]
        i += 1

# useful topics are 'lsst.sal.X' 
print(list(topic_dict['lsst']['sal'].keys()))

In [ ]:
# Pick a CSC and see more about its topics
ss = 'MTAOS'
for tt in topic_dict['lsst']['sal'][ss].keys():
    topic = f'lsst.sal.{ss}.{tt}'
    print(topic)
    try:
        fields = await client.get_fields(topic)
    except: 
        fields  = ['value']
    fields = [f for f in fields if ('private' not in f) and (f != 'name') and (f != 'duration')]
    try:
        dd = await client.select_top_n(topic, fields, 1)
    except: 
        print('query failed')
    display(dd)    
	

An example used in Craig's [notebook](https://rubin-obs.slack.com/files/U07NQHE622U/F07UYPV2XDY/comcam_get_aos_for_exposure_05nov24.ipynb)  employs `lsst.sal.MtM1M3.appliedForces` and `lsst.sal.MTHexapod.application`. We can query these topics within the specified time:

In [ ]:
t1 = expRecordBegin.timespan.begin.utc
t2 = expRecordEnd.timespan.end.utc
end_readout = await client.select_time_series("lsst.sal.MTM1M3.appliedForces", '*',t1, t2)


In [ ]:
end_readout[:5]

In [ ]:
hexData = await client.select_time_series("lsst.sal.MTHexapod.application", '*',t1,t2)
print(len(hexData))

In [ ]:
hexData[:5]

In [ ]:
m2Hex = hexData[hexData['salIndex'] == 2]
print(len(m2Hex))

In [ ]:
m2Hex[:5]

In [ ]:
m2Hex.columns

Plot the x,y,z position as a function of time : 

In [ ]:
X = m2Hex["position0"]
Y = m2Hex["position1"]
Z = m2Hex["position2"]
time = m2Hex.index

In [ ]:
import matplotlib.pyplot as plt 
fig,ax = plt.subplots(3,1,figsize=(12,12))
i = 0 
for pos, name in zip([X,Y,Z], 'xyz'):
    ax[i].plot(time, pos, )
    ax[i].set_ylabel(f'{name} position '+r'[$\mu $m]')
    
    ax[i].axvspan(expRecordBegin.timespan.begin.utc.datetime, 
               expRecordBegin.timespan.end.utc.datetime, 
               alpha=0.5, color='red')

    ax[i].axvspan(expRecordEnd.timespan.begin.utc.datetime, 
               expRecordEnd.timespan.end.utc.datetime, 
               alpha=0.5, color='blue')


    
    i += 1 
ax[-1].set_xlabel('Time [h:m:s]')
ax[0].set_title(f'MT hexapod position, exposures {expRecordBegin.id} and {expRecordBegin.id}')

The two filled rectangles mark the integration time of the two considered  exposures.